# IBM HR Analytics Employee Attrition Dataset

Synthetisch generierter Datensatz von IBM Data Scientists.
Ziel: Faktoren identifizieren, die zu Mitarbeiterkündigung führen.

## Kodierung ordinaler Variablen

| Wert | Education     | EnvironmentSatisfaction | JobInvolvement | JobSatisfaction | PerformanceRating | RelationshipSatisfaction | WorkLifeBalance |
| ---- | ------------- | ----------------------- | -------------- | --------------- | ----------------- | ------------------------ | --------------- |
| 1    | Below College | Low                     | Low            | Low             | Low               | Low                      | Bad             |
| 2    | College       | Medium                  | Medium         | Medium          | Good              | Medium                   | Good            |
| 3    | Bachelor      | High                    | High           | High            | Excellent         | High                     | Better          |
| 4    | Master        | Very High               | Very High      | Very High       | Outstanding       | Very High                | Best            |
| 5    | Doctor        | —                       | —              | —               | —                 | —                        | —               |


# Ablauf einer EDA an Rahmanys Schritten:


In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings as wr
import hashlib
from pathlib import Path
import datetime
wr.filterwarnings('ignore')

## Schritt 1 - Formale Erfassung


In [23]:
pfad = Path('data/ibm_original.csv')
df = pd.read_csv(pfad)
df

,Age,Attrition,BusinessTravel,DailyRate,Department,DistanceFromHome,Education,EducationField,EmployeeCount,EmployeeNumber,...,RelationshipSatisfaction,StandardHours,StockOptionLevel,TotalWorkingYears,TrainingTimesLastYear,WorkLifeBalance,YearsAtCompany,YearsInCurrentRole,YearsSinceLastPromotion,YearsWithCurrManager
0,41,Yes,Travel_Rarely,1102,Sales,1,2,Life Sciences,1,1,...,1,80,0,8,0,1,6,4,0,5
1,49,No,Travel_Frequently,279,Research & Development,8,1,Life Sciences,1,2,...,4,80,1,10,3,3,10,7,1,7
2,37,Yes,Travel_Rarely,1373,Research & Development,2,2,Other,1,4,...,2,80,0,7,3,3,0,0,0,0
3,33,No,Travel_Frequently,1392,Research & Development,3,4,Life Sciences,1,5,...,3,80,0,8,3,3,8,7,3,0
4,27,No,Travel_Rarely,591,Research & Development,2,1,Medical,1,7,...,4,80,1,6,3,3,2,2,2,2
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1465,36,No,Travel_Frequently,884,Research & Development,23,2,Medical,1,2061,...,3,80,1,17,3,3,5,2,0,3
1466,39,No,Travel_Rarely,613,Research & Development,6,1,Medical,1,2062,...,1,80,1,9,5,3,7,7,1,7
1467,27,No,Travel_Rarely,155,Research & Development,4,3,Life Sciences,1,2064,...,2,80,1,6,0,3,6,2,0,3
1468,49,No,Travel_Frequently,1023,Sales,2,3,Medical,1,2065,...,4,80,0,17,3,2,9,6,0,8


## Hash-Dokumentation

use sha256() to create a SHA-256 hash object


In [3]:
datei_hash = hashlib.sha256(pfad.read_bytes()).hexdigest() #https://docs.python.org/3/library/hashlib.html

doku = {
    'datei' : pfad.name,
    'sha256' : datei_hash,
    'zeilen_roh' : df.shape[0],
    'spalten_roh' : df.shape[1],
    'pandas_version' : pd.__version__,
    'numpy_version' : np.__version__,
    'erstellt_am' : datetime.date.today().isoformat(),

}

for k, v in doku.items():
    print(f"{k}: {v}")

# Doku persistent ablegen
import json
with open('data/datensatz_meta.json', 'w') as f:
    json.dump(doku, f, indent=2, ensure_ascii=False)

datei: ibm_original.csv
sha256: a5c31e38bd7fafc9bc333884eb181b06b41b8e5e488e8f7ccb27199fb3be7659
zeilen_roh: 1470
spalten_roh: 35
pandas_version: 3.0.3
numpy_version: 2.4.6
erstellt_am: 2026-06-26


Data Frame Check: Für die Analyse werden nichtaussagende Prädikatoren aus dem Frame entfernt für eine bereinigte Sicht


In [4]:
null_variance_cols = ['EmployeeCount', 'StandardHours', 'Over18']

print ("DataFrame Shape vor dem Check:", df.shape)
for col in null_variance_cols:
    unique_vals = df[col].unique()
    print(f"Spalte '{col}' hat Konstanz: {unique_vals} (Anzahl eindeutiger Werte: {len(unique_vals)})")

# nutzlose Prädikatoren nicht droppen, um später zu prüfen, ob LLM diese selbstständig erkennt

# Separate, bereinigte Sicht NUR für eigene Referenzwerte

df_referenz = df.drop(columns=null_variance_cols)

DataFrame Shape vor dem Check: (1470, 35)
Spalte 'EmployeeCount' hat Konstanz: [1] (Anzahl eindeutiger Werte: 1)
Spalte 'StandardHours' hat Konstanz: [80] (Anzahl eindeutiger Werte: 1)
Spalte 'Over18' hat Konstanz: <StringArray>
['Y']
Length: 1, dtype: str (Anzahl eindeutiger Werte: 1)


In [5]:
# Konstante Variablen entfernen (keine Varianz, kein Analysewert)

#df.drop(columns=['EmployeeCount', 'StandardHours', 'Over18'])

In [6]:
df_referenz.shape

(1470, 32)

In [7]:
df_referenz.info()

<class 'pandas.DataFrame'>
RangeIndex: 1470 entries, 0 to 1469
Data columns (total 32 columns):
 #   Column                    Non-Null Count  Dtype
---  ------                    --------------  -----
 0   Age                       1470 non-null   int64
 1   Attrition                 1470 non-null   str  
 2   BusinessTravel            1470 non-null   str  
 3   DailyRate                 1470 non-null   int64
 4   Department                1470 non-null   str  
 5   DistanceFromHome          1470 non-null   int64
 6   Education                 1470 non-null   int64
 7   EducationField            1470 non-null   str  
 8   EmployeeNumber            1470 non-null   int64
 9   EnvironmentSatisfaction   1470 non-null   int64
 10  Gender                    1470 non-null   str  
 11  HourlyRate                1470 non-null   int64
 12  JobInvolvement            1470 non-null   int64
 13  JobLevel                  1470 non-null   int64
 14  JobRole                   1470 non-null   str  
 15

## Schritt 2

Univariate ANalyse über describe() und Häufigkeiten


In [8]:
df_referenz.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,1470.0,36.923810,9.135373,18.0,30.00,36.0,43.00,60.0
DailyRate,1470.0,802.485714,403.509100,102.0,465.00,802.0,1157.00,1499.0
DistanceFromHome,1470.0,9.192517,8.106864,1.0,2.00,7.0,14.00,29.0
Education,1470.0,2.912925,1.024165,1.0,2.00,3.0,4.00,5.0
EmployeeNumber,1470.0,1024.865306,602.024335,1.0,491.25,1020.5,1555.75,2068.0
EnvironmentSatisfaction,1470.0,2.721769,1.093082,1.0,2.00,3.0,4.00,4.0
HourlyRate,1470.0,65.891156,20.329428,30.0,48.00,66.0,83.75,100.0
JobInvolvement,1470.0,2.729932,0.711561,1.0,2.00,3.0,3.00,4.0
JobLevel,1470.0,2.063946,1.106940,1.0,1.00,2.0,3.00,5.0
JobSatisfaction,1470.0,2.728571,1.102846,1.0,2.00,3.0,4.00,4.0


In [9]:
df_referenz.columns.tolist()

['Age',
 'Attrition',
 'BusinessTravel',
 'DailyRate',
 'Department',
 'DistanceFromHome',
 'Education',
 'EducationField',
 'EmployeeNumber',
 'EnvironmentSatisfaction',
 'Gender',
 'HourlyRate',
 'JobInvolvement',
 'JobLevel',
 'JobRole',
 'JobSatisfaction',
 'MaritalStatus',
 'MonthlyIncome',
 'MonthlyRate',
 'NumCompaniesWorked',
 'OverTime',
 'PercentSalaryHike',
 'PerformanceRating',
 'RelationshipSatisfaction',
 'StockOptionLevel',
 'TotalWorkingYears',
 'TrainingTimesLastYear',
 'WorkLifeBalance',
 'YearsAtCompany',
 'YearsInCurrentRole',
 'YearsSinceLastPromotion',
 'YearsWithCurrManager']

In [10]:
df_referenz.isnull().sum()

Age                         0
Attrition                   0
BusinessTravel              0
DailyRate                   0
Department                  0
DistanceFromHome            0
Education                   0
EducationField              0
EmployeeNumber              0
EnvironmentSatisfaction     0
Gender                      0
HourlyRate                  0
JobInvolvement              0
JobLevel                    0
JobRole                     0
JobSatisfaction             0
MaritalStatus               0
MonthlyIncome               0
MonthlyRate                 0
NumCompaniesWorked          0
OverTime                    0
PercentSalaryHike           0
PerformanceRating           0
RelationshipSatisfaction    0
StockOptionLevel            0
TotalWorkingYears           0
TrainingTimesLastYear       0
WorkLifeBalance             0
YearsAtCompany              0
YearsInCurrentRole          0
YearsSinceLastPromotion     0
YearsWithCurrManager        0
dtype: int64

Es wurden keine MissingValues gefunden


In [11]:
df_referenz.duplicated().sum()

np.int64(0)

In [12]:
for column in df_referenz.columns:
    print(f"{column}: Number of unique values {df_referenz[column].nunique()}")
    print("==========================================================")

Age: Number of unique values 43
Attrition: Number of unique values 2
BusinessTravel: Number of unique values 3
DailyRate: Number of unique values 886
Department: Number of unique values 3
DistanceFromHome: Number of unique values 29
Education: Number of unique values 5
EducationField: Number of unique values 6
EmployeeNumber: Number of unique values 1470
EnvironmentSatisfaction: Number of unique values 4
Gender: Number of unique values 2
HourlyRate: Number of unique values 71
JobInvolvement: Number of unique values 4
JobLevel: Number of unique values 5
JobRole: Number of unique values 9
JobSatisfaction: Number of unique values 4
MaritalStatus: Number of unique values 3
MonthlyIncome: Number of unique values 1349
MonthlyRate: Number of unique values 1427
NumCompaniesWorked: Number of unique values 10
OverTime: Number of unique values 2
PercentSalaryHike: Number of unique values 15
PerformanceRating: Number of unique values 2
RelationshipSatisfaction: Number of unique values 4
StockOptio

## Berechnung der Referenzwerte:

Anzahl der richtigen Antworten, die später ChatGPT gefragt wird um zu vergleichen, ob dasselbe rauskommt oder was anderes


In [13]:
# 1. Attrition bedeutet, wie viele Menschen die Firma verlassen haben.

df_referenz['Attrition'].value_counts()


Attrition
No     1233
Yes     237
Name: count, dtype: int64

In [14]:
df_referenz['Attrition'].value_counts(normalize=True)


Attrition
No     0.838776
Yes    0.161224
Name: proportion, dtype: float64

## Kategoriale Merkmale: Verteilung

Randhäufigkeiten je Kategorie, um die Zusammensetzung des Datensatzes zu erfassen (z.B. Geschlechterverhältnis, Verteilung über Abteilungen). Dient als Referenzwert für einfache Zählfragen an LLM. Die für die Forschungsfrage zentrale Kündigungsquote pro Kategorie wird separat berechnet, da value_counts() nur die Gruppengröße zeigt, nicht den Zusammenhang mit Attrition.


In [15]:
# Nominale Merkmale (ohne Zielvariable Attrition, ohne ordinale Codes, die bereits numerisch in die Korrelation eingehen)

kategorisch = ['Gender', 'Department', 'JobRole', 
               'MaritalStatus', 'BusinessTravel',
               'EducationField', 'OverTime']

for col in kategorisch:
    print(col)
    print(df_referenz[col].value_counts())
    print('---')

Gender
Gender
Male      882
Female    588
Name: count, dtype: int64
---
Department
Department
Research & Development    961
Sales                     446
Human Resources            63
Name: count, dtype: int64
---
JobRole
JobRole
Sales Executive              326
Research Scientist           292
Laboratory Technician        259
Manufacturing Director       145
Healthcare Representative    131
Manager                      102
Sales Representative          83
Research Director             80
Human Resources               52
Name: count, dtype: int64
---
MaritalStatus
MaritalStatus
Married     673
Single      470
Divorced    327
Name: count, dtype: int64
---
BusinessTravel
BusinessTravel
Travel_Rarely        1043
Travel_Frequently     277
Non-Travel            150
Name: count, dtype: int64
---
EducationField
EducationField
Life Sciences       606
Medical             464
Marketing           159
Technical Degree    132
Other                82
Human Resources      27
Name: count, dtype: int64

In [16]:
# Overtime in Korrelation miteinbringen, weil es ein String ist


df_referenz['OverTime_num'] = df_referenz['OverTime'].map({'Yes': 1, 'No': 0})


In [17]:
# 2. Grundstatistik für alle Zahlenspalten

df_referenz.describe().T

,count,mean,std,min,25%,50%,75%,max
Age,1470.0,36.923810,9.135373,18.0,30.00,36.0,43.00,60.0
DailyRate,1470.0,802.485714,403.509100,102.0,465.00,802.0,1157.00,1499.0
DistanceFromHome,1470.0,9.192517,8.106864,1.0,2.00,7.0,14.00,29.0
Education,1470.0,2.912925,1.024165,1.0,2.00,3.0,4.00,5.0
EmployeeNumber,1470.0,1024.865306,602.024335,1.0,491.25,1020.5,1555.75,2068.0
EnvironmentSatisfaction,1470.0,2.721769,1.093082,1.0,2.00,3.0,4.00,4.0
HourlyRate,1470.0,65.891156,20.329428,30.0,48.00,66.0,83.75,100.0
JobInvolvement,1470.0,2.729932,0.711561,1.0,2.00,3.0,3.00,4.0
JobLevel,1470.0,2.063946,1.106940,1.0,1.00,2.0,3.00,5.0
JobSatisfaction,1470.0,2.728571,1.102846,1.0,2.00,3.0,4.00,4.0


## Schritt 3 - Korrelation mit Attrition


In [18]:
# 3. Berechnung Attrition als Zahl

df_referenz['Attrition_num'] = df_referenz['Attrition'].map({'Yes' : 1, 'No' : 0})

In [19]:
# 4. Welche Variablen hängen mit Kündigungen zusammen

df_referenz.corr(numeric_only=True)['Attrition_num'].sort_values() #numeric_only rechnet Pearson Korrelation

TotalWorkingYears          -0.171063
JobLevel                   -0.169105
YearsInCurrentRole         -0.160545
MonthlyIncome              -0.159840
Age                        -0.159205
YearsWithCurrManager       -0.156199
StockOptionLevel           -0.137145
YearsAtCompany             -0.134392
JobInvolvement             -0.130016
JobSatisfaction            -0.103481
EnvironmentSatisfaction    -0.103369
WorkLifeBalance            -0.063939
TrainingTimesLastYear      -0.059478
DailyRate                  -0.056652
RelationshipSatisfaction   -0.045872
YearsSinceLastPromotion    -0.033019
Education                  -0.031373
PercentSalaryHike          -0.013478
EmployeeNumber             -0.010577
HourlyRate                 -0.006846
PerformanceRating           0.002889
MonthlyRate                 0.015170
NumCompaniesWorked          0.043494
DistanceFromHome            0.077924
OverTime_num                0.246118
Attrition_num               1.000000
Name: Attrition_num, dtype: float64

+1 = steigt zusammen
0 = kein Zusammenhang
-1 gegenläufig


## Schritt 4 - Auffällige Muster wie fehlende/inkonsistente Werte


## Schritt 5 - Ausreißererkennung

Erkennung von Ausrißern in den metrischen Variablen mittels IQR-Methode als Baseline. Als Ausreißer gilt ein Wert außerhalb von [Q1 - 1,5*IQR, Q3 + 1,5*IQR] (https://online.stat.psu.edu/stat200/lesson/3/3.2). Die Anzahl je Variable wird als Referenzwert mitgenommen. Ausreißer sollen nur erkannt, aber nicht entfernt werden.


In [ ]:
# Metrische Variablen (echte Messwerte, keine ordinalen Codes, keine Hilfsspalten)

metrisch = ['Age', 'MonthlyIncome', 'DailyRate', 'MonthlyRate', 'HourlyRate',
            'DistanceFromHome', 'TotalWorkingYears', 'YearsAtCompany',
            'YearsInCurrentRole', 'YearsSinceLastPromotion',
            'YearsWithCurrManager', 'NumCompaniesWorked', 'PercentSalaryHike',
            'TrainingTimesLastYear']

ausreisser = {}
for col in metrisch:
    Q1 = df_referenz[col].quantile(0.25)
    Q3 = df_referenz[col].quantile(0.75)
    IQR = Q3-Q1
    untere = Q1 - 1.5* IQR # unter erstes Quantil
    obere = Q3 + 1.5 * IQR # über drittes Quantil

    maske = ((df_referenz[col] < untere) | (df_referenz[col] > obere)) #Wahrheitsfaktor, der für jede Zelle sagt, ob sie außerhalb liegt
    anzahl = maske.sum()

    ausreisser[col] = {
        'anzahl': int(anzahl),
        'anteil_prozent': round(100 * anzahl / len(df_referenz), 2),
        'untere_grenze': round(untere, 2),
        'obere_grenze' : round(obere, 2)
    }

ausreisser_df = pd.DataFrame(ausreisser).T.sort_values('anzahl', ascending=False)
ausreisser_df

,anzahl,anteil_prozent,untere_grenze,obere_grenze
TrainingTimesLastYear,238.0,16.19,0.50,4.50
MonthlyIncome,114.0,7.76,-5291.00,16581.00
YearsSinceLastPromotion,107.0,7.28,-4.50,7.50
YearsAtCompany,104.0,7.07,-6.00,18.00
TotalWorkingYears,63.0,4.29,-7.50,28.50
NumCompaniesWorked,52.0,3.54,-3.50,8.50
YearsInCurrentRole,21.0,1.43,-5.50,14.50
YearsWithCurrManager,14.0,0.95,-5.50,14.50
DistanceFromHome,0.0,0.00,-16.00,32.00
DailyRate,0.0,0.00,-573.00,2195.00


## Export


In [24]:
# 1. Numerische Werte

df_referenz.describe().T.to_csv('data/referenzwerte_numerisch.csv')

numeric_only rechnet Pearson und mischt dabei echte metrische Größen (Alter, Einkommen) mit ordinalen Skalen (Education 1-5, die Satisfaction-Werte). Pearson unterstellt lineare Zusammenhänge zwischen intervallskalierten Variablen. Es ist bewusst als einfache Baseline, aber die Limitationen sind bekannt.


In [25]:
# 2. Korrelationen 

korr = df_referenz.corr(numeric_only=True)['Attrition_num'].sort_values()
korr = korr.drop('Attrition_num') #Attrition_num = 1.0 rausfiltern
korr.to_csv('data/korrelationen.csv')


In [26]:
# 3. Attrition_Rate

for col in kategorisch:
    df_referenz[col].value_counts().to_csv(f'data/haeufigkeiten_{col}.csv')

In [27]:
# 4. Ausreißer

ausreisser_df.to_csv('data/ausreisser.csv')